In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
import lightgbm as lgb

In [ ]:
train = pd.read_csv("/kaggle/input/mldataset/train.csv")
test = pd.read_csv("/kaggle/input/mldataset/test.csv")

train = train.dropna(subset=["catalog_content", "price"])
test = test.fillna({"catalog_content": ""})

In [ ]:
class NumericExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        feats = []
        for text in X:
            t = text.lower()
            nums = re.findall(r"(\d+\.?\d*)", t)
            nums = [float(n) for n in nums] if nums else [0]

            weight_oz = re.findall(r"(\d+\.?\d*)\s?oz", t)
            weight_lb = re.findall(r"(\d+\.?\d*)\s?lb", t)
            weight_g = re.findall(r"(\d+\.?\d*)\s?g", t)
            weight_ml = re.findall(r"(\d+\.?\d*)\s?ml", t)

            feats.append({
                "avg_num": np.mean(nums),
                "max_num": np.max(nums),
                "num_count": len(nums),
                "weight_oz": float(weight_oz[0]) if weight_oz else 0,
                "weight_lb": float(weight_lb[0]) if weight_lb else 0,
                "weight_g": float(weight_g[0]) if weight_g else 0,
                "weight_ml": float(weight_ml[0]) if weight_ml else 0,
            })
        return feats


In [ ]:
tfidf = TfidfVectorizer(
    max_features=80000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=3
)

num_pipe = Pipeline([
    ('extract', NumericExtractor()),
    ('vectorize', DictVectorizer(sparse=True))
])

combined = FeatureUnion([
    ('tfidf', tfidf),
    ('num', num_pipe)
])

In [ ]:
X_all = combined.fit_transform(train["catalog_content"])
X_test_all = combined.transform(test["catalog_content"])
y = train["price"]


y_log = np.log1p(y)

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_log, test_size=0.2, random_state=42
)


In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 3,
    "min_data_in_leaf": 30,
    "verbose": -1,
    "random_state": 42
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

model = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

In [ ]:
val_pred_log = model.predict(X_val)
val_pred = np.expm1(val_pred_log)
y_val_exp = np.expm1(y_val)

def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0
    return 100 * np.mean(diff)

val_smape = smape(y_val_exp, val_pred)
print(f"\n Validation SMAPE: {val_smape:.2f}")

In [ ]:
final_model = lgb.train(
    params,
    lgb.Dataset(X_all, label=y_log),
    num_boost_round=int(model.best_iteration * 1.1)
)

test_pred_log = final_model.predict(X_test_all)
test_pred = np.clip(np.expm1(test_pred_log), 0, None)

In [ ]:
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": test_pred
})
submission.to_csv("submission_lgbm_t.csv", index=False)
print(" submission_lgbm_t.csv saved!")